In [38]:
import os
import sys

# Konfiguracja ścieżek Kaggle
WORKING_DIR = '/kaggle/working'
INPUT_PATH = '/kaggle/input/datasets/maciejjakubowski/custom-dataset'
REPO_NAME = 'e2e-climbing-vision'
REPO_URL = f'https://github.com/jeicam3/{REPO_NAME}'
BRANCH_NAME = 'devel'

# Przejście do katalogu roboczego
os.chdir(WORKING_DIR)

# Klonowanie / aktualizacja repozytorium z GitHub
if not os.path.exists(f'{WORKING_DIR}/{REPO_NAME}'):
    print(f"📥 Klonowanie repozytorium (branch: {BRANCH_NAME})...")
    !git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"🔄 Aktualizacja repozytorium (branch: {BRANCH_NAME})...")
    !git -C {WORKING_DIR}/{REPO_NAME} fetch origin
    !git -C {WORKING_DIR}/{REPO_NAME} checkout {BRANCH_NAME}
    !git -C {WORKING_DIR}/{REPO_NAME} pull origin {BRANCH_NAME}

# Dodanie repozytorium do ścieżki Python, aby importy działały
repo_full_path = os.path.join(WORKING_DIR, REPO_NAME)
if repo_full_path not in sys.path:
    sys.path.append(repo_full_path)

print(f"\n✅ Środowisko Kaggle jest gotowe do pracy.")

🔄 Aktualizacja repozytorium (branch: devel)...
Already on 'devel'
Your branch is up to date with 'origin/devel'.
From https://github.com/jeicam3/e2e-climbing-vision
 * branch            devel      -> FETCH_HEAD
Already up to date.

✅ Środowisko Kaggle jest gotowe do pracy.


In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd
import copy

# Importy struktur z Twojego repozytorium
from models.dataset import ClimbingDataset
from models.efficientnet import get_climbing_model, update_model_freezing

CHOSEN_MODEL = "b2"          # "b0", "b1", "b2"
FREEZE_START = 6             # Od jakiego bloku zaczynamy Fazę 1 (8 = tylko klasyfikator)
BATCH_SIZE = 32              # rozmiar batcha
DROPOUT_RATE = 0.6           # Regularyzacja klasyfikatora

# 1. Inicjalizacja wybranego modelu i pobranie jego rozdzielczości
model, target_resolution = get_climbing_model(
    model_name=CHOSEN_MODEL,
    num_classes=4,
    dropout_rate=DROPOUT_RATE,
    freeze_until_block=FREEZE_START
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 2. Definicja transformacji danych (z uwzględnieniem dynamicznego wymiaru)
train_transforms = transforms.Compose([
    transforms.Resize((target_resolution, target_resolution)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

val_transforms = transforms.Compose([
    transforms.Resize((target_resolution, target_resolution)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Przygotowanie podziału danych na podstawie filmów (Strict Split)
CSV_PATH = f'{INPUT_PATH}/dataset/labels.csv'
IMG_DIR = f'{INPUT_PATH}/dataset/full-data'
test_videos = ['p9_orange', 'p9_green', 'p4_orange', 'p4_green', 'IMG_0903', 'IMG_0895', 'p5_orange', 'p5_green', 'p10_green', 'p10_orange']

df = pd.read_csv(CSV_PATH)
val_mask = df.iloc[:, 0].str.contains('|'.join(test_videos))

train_df = df[~val_mask].reset_index(drop=True)
val_df = df[val_mask].reset_index(drop=True)

train_df.to_csv('train_labels_split.csv', index=False)
val_df.to_csv('val_labels_split.csv', index=False)

# 4. Tworzenie Loaderów
train_data = ClimbingDataset(csv_file='train_labels_split.csv', img_dir=IMG_DIR, transform=train_transforms)
val_data = ClimbingDataset(csv_file='val_labels_split.csv', img_dir=IMG_DIR, transform=val_transforms)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n📊 Podział danych zakończony na urządzeniu: {device}")
print(f"📊 Próbki treningowe: {len(train_data)} | Walidacyjne: {len(val_data)}")

Załadowano EfficientNet-B2 (Natywna rozdzielczość: 260x260)

📊 Podział danych zakończony na urządzeniu: cuda
📊 Próbki treningowe: 3273 | Walidacyjne: 894


In [40]:
TWO_STAGE_TRAINING = True     # True = Wieloetapowy | False = Klasyczny trening (Jednoetapowy)
SWITCH_EPOCH = 13             # Faza 2 zaczyna się od tej epoki (włącznie)
FREEZE_STAGE_2 = 4            # Do którego bloku zamrażamy w Fazie 2

TOTAL_EPOCHS = 40
PATIENCE_PLATEAU = 5
LABEL_SMOOTHING = 0.05

# Konfiguracja początkowa (Faza 1)
START_LR = 0.0001
WEIGHT_DECAY_F1 = 3e-3
SCHEDULER_TYPE_F1 = "one_cycle" # "one_cycle", "plateau", "cosine"

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=START_LR, weight_decay=WEIGHT_DECAY_F1)

# Funkcja generująca scheduler w zależności od potrzeb i fazy
def build_scheduler(opt, sched_type, epochs_budget):
    if sched_type == "one_cycle":
        steps = len(train_loader) * epochs_budget
        return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.001, total_steps=steps, pct_start=0.3), True
    elif sched_type == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=4, factor=0.5), False
    elif sched_type == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10), False

current_scheduler_type = SCHEDULER_TYPE_F1
scheduler, step_per_batch = build_scheduler(optimizer, current_scheduler_type, TOTAL_EPOCHS)

# Nazwy plików wyjściowych (zbudowane automatycznie na podstawie konfiguracji)
prefix = "two_stage" if TWO_STAGE_TRAINING else "single_stage"
MODEL_NAME = f"{prefix}_{CHOSEN_MODEL}_run.pth"
LOG_NAME = f"{prefix}_{CHOSEN_MODEL}_run.txt"

# Inicjalizacja logów tekstowych
history_logs = []
history_logs.append(f"=== REJESTR EKSPERYMENTU: {prefix.upper()} ===\n")
history_logs.append(f"Model: EfficientNet-{CHOSEN_MODEL.upper()} | Resolution: {target_resolution}x{target_resolution}\n")
history_logs.append(f"Faza 1 -> Freeze: {FREEZE_START}, Scheduler: {SCHEDULER_TYPE_F1}, Start LR: {START_LR}\n")
if TWO_STAGE_TRAINING:
    history_logs.append(f"Faza 2 -> Switch Epoch: {SWITCH_EPOCH}, New Freeze: {FREEZE_STAGE_2}\n")
history_logs.append("-" * 60 + "\n")
history_logs.append("Epoch | LR | Train Loss | Val Loss\n")

# Zmienne monitorujące stan treningu
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
early_stop_counter = 0

print(f"🚀 Uruchamianie silnika treningowego na: {device}")
print("-" * 50)

for epoch in range(TOTAL_EPOCHS):
    current_epoch_idx = epoch + 1
    
    # --- LOGIKA PRZEŁĄCZANIA NA FAZĘ 2 ---
    if TWO_STAGE_TRAINING and current_epoch_idx == SWITCH_EPOCH:
        print(f"\n⚡ === PRZEŁĄCZANIE PARAMETRÓW: ROZPOCZYNANIE FAZY 2 ===")
        
        # 1. Częściowe uwolnienie wag sieci bazowej
        update_model_freezing(model, freeze_until_block=FREEZE_STAGE_2)
        
        # 2. Re-inicjalizacja optymalizatora ze znacznie bezpieczniejszym, mikro-kontrolnym LR i wyższym WD
        optimizer = optim.Adam(model.parameters(), lr=5e-5, weight_decay=5e-3)
        
        # 3. Zmiana Schedulera na Plateau w celu dokładnego "oszlifowania" uzyskanej doliny
        current_scheduler_type = "plateau"
        remaining_epochs = TOTAL_EPOCHS - epoch
        scheduler, step_per_batch = build_scheduler(optimizer, current_scheduler_type, remaining_epochs)
    
    # --- ETAP TRENINGU (TRAIN) ---
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()
        
        # Aplikacja Label Smoothingu
        labels = labels * (1 - 2 * LABEL_SMOOTHING) + LABEL_SMOOTHING
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        if step_per_batch:
            scheduler.step()
            
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    # --- ETAP WALIDACJI (VALIDATION) ---
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model(images)
            v_loss = criterion(outputs, labels)
            running_val_loss += v_loss.item()

    avg_val_loss = running_val_loss / len(val_loader)

    # Krok schedulera sterowanego epokami (Plateau lub Cosine)
    if not step_per_batch:
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(avg_val_loss)
        else:
            scheduler.step()
            
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{current_epoch_idx:02d}/{TOTAL_EPOCHS}] | LR: {current_lr:.6f} | Loss: T {avg_train_loss:.4f} / V {avg_val_loss:.4f}")
    history_logs.append(f"{current_epoch_idx:02d} | {current_lr:.6f} | {avg_train_loss:.4f} | {avg_val_loss:.4f}\n")

    # --- KONTROLA I ZAPIS REKORDÓW ---
    if avg_val_loss < best_val_loss:
        print(f"⭐ Nowy najlepszy wynik walidacji!")
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), MODEL_NAME)
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        # Early stopping chroni model tylko w fazie stabilizacji (Plateau)
        if current_scheduler_type == "plateau":
            if early_stop_counter >= PATIENCE_PLATEAU:
                print(f"\n🛑 EARLY STOPPING wywołany w epoce {current_epoch_idx}.")
                break

# Przywrócenie najlepszych wag z całego procesu treningowego
model.load_state_dict(best_model_wts)
print("-" * 50 + f"\n🏁 Trening zakończony. Najlepsza uzyskana strata walidacji: {best_val_loss:.4f}")

🚀 Uruchamianie silnika treningowego na: cuda
--------------------------------------------------
Epoch [01/40] | LR: 0.000056 | Loss: T 0.6217 / V 0.5620
⭐ Nowy najlepszy wynik walidacji!
Epoch [02/40] | LR: 0.000104 | Loss: T 0.5068 / V 0.4701
⭐ Nowy najlepszy wynik walidacji!
Epoch [03/40] | LR: 0.000181 | Loss: T 0.4664 / V 0.4764
Epoch [04/40] | LR: 0.000280 | Loss: T 0.4466 / V 0.4858
Epoch [05/40] | LR: 0.000396 | Loss: T 0.4367 / V 0.4960
Epoch [06/40] | LR: 0.000521 | Loss: T 0.4381 / V 0.4388
⭐ Nowy najlepszy wynik walidacji!
Epoch [07/40] | LR: 0.000645 | Loss: T 0.4367 / V 0.4522
Epoch [08/40] | LR: 0.000761 | Loss: T 0.4380 / V 0.5552
Epoch [09/40] | LR: 0.000860 | Loss: T 0.4346 / V 0.4906
Epoch [10/40] | LR: 0.000936 | Loss: T 0.4361 / V 0.4332
⭐ Nowy najlepszy wynik walidacji!
Epoch [11/40] | LR: 0.000984 | Loss: T 0.4396 / V 0.4832
Epoch [12/40] | LR: 0.001000 | Loss: T 0.4297 / V 0.4851

⚡ === PRZEŁĄCZANIE PARAMETRÓW: ROZPOCZYNANIE FAZY 2 ===
Epoch [13/40] | LR: 0.00005

In [1]:
import os

# Definiowanie unikalnego podfolderu dla danej próby w katalogu roboczym Kaggle
SAVE_DIR = "/kaggle/working/second_custom"
os.makedirs(SAVE_DIR, exist_ok=True)

# Zapis przygotowanych logów historycznych (.txt)
log_full_path = os.path.join(SAVE_DIR, LOG_NAME)
with open(log_full_path, "w") as f:
    f.writelines(history_logs)

# Zapis ostatecznego, najlepszego pliku wag (.pth)
model_full_path = os.path.join(SAVE_DIR, MODEL_NAME)
torch.save(model.state_dict(), model_full_path)

print(f"   📊 Logi tekstowe: {log_full_path}")
print(f"   🧠 Wagi modelu:   {model_full_path}")

NameError: name 'LOG_NAME' is not defined